In [1]:
# ==============================================================================
# PASO 1: INSTALACIÓN Y PREPARACIÓN DE BIG DATA PARA DRL
# ==============================================================================
!pip install gymnasium stable-baselines3 shimmy -q

import pandas as pd
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, auc, f1_score, recall_score, matthews_corrcoef, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from google.colab import drive

drive.mount('/content/drive')

print("🚀 1. Cargando matriz gigante para DRL...")
ruta_datos = '/content/drive/MyDrive/Colab Notebooks/MAESTRIA IA/PROYECTO - DESARROLLO DE SOLUCIONES/dataset_limpio_IA.csv'

# Compresión de memoria vital para 1.4M de registros
df = pd.read_csv(ruta_datos)
for col in df.columns:
    if df[col].dtype == 'float64': df[col] = df[col].astype('float32')
    if df[col].dtype == 'int64': df[col] = df[col].astype('int32')

col_objetivo = 'FRAUDE_IA'
Y = df[col_objetivo].values
X = df.drop(columns=[col_objetivo]).values

del df
gc.collect()

print("⚙️ 2. Particionando y Escalando (Obligatorio para DRL)...")
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==============================================================================
# PASO 2: CREACIÓN DEL ENTORNO BANCARIO (CON OPTION LEARNING)
# ==============================================================================
print("\n🏦 3. Construyendo el Entorno de Simulacion Bancaria...")

class EntornoBancoBigData(gym.Env):
    def __init__(self, X_data, Y_data):
        super(EntornoBancoBigData, self).__init__()
        self.X_data = X_data
        self.Y_data = Y_data

        # 3 Acciones: 0 = Permitir, 1 = Bloquear, 2 = Solicitar OTP (Fricción suave)
        self.action_space = spaces.Discrete(3)
        self.observation_space = spaces.Box(low=0, high=1, shape=(X_data.shape[1],), dtype=np.float32)
        self.current_step = 0

    def step(self, action):
        is_fraud = self.Y_data[self.current_step]

        # SISTEMA DE RECOMPENSAS (Option Learning)
        if is_fraud == 1:
            if action == 1: reward = 10     # Excelente: Bloqueó un fraude
            elif action == 2: reward = 5    # Bueno: Pidió OTP a un fraude (Option Learning)
            else: reward = -100             # ERROR FATAL: Dejó pasar un fraude
        else:
            if action == 0: reward = 1      # Excelente: Dejó pasar un cliente sano
            elif action == 2: reward = -2   # Leve molestia: Pidió OTP a cliente sano
            else: reward = -10              # ERROR: Bloqueó a un cliente sano (Falso Positivo)

        self.current_step += 1
        done = self.current_step >= len(self.X_data) - 1

        if not done:
            obs = self.X_data[self.current_step]
        else:
            obs = self.X_data[0] # Reiniciar si se acaba

        return obs, reward, done, False, {}

    def reset(self, seed=None):
        self.current_step = 0
        return self.X_data[0], {}

# ==============================================================================
# PASO 3: ENTRENAMIENTO DEL AGENTE PPO
# ==============================================================================
# Para no tardar horas, entrenamos sobre una muestra de 100,000 transacciones
limite_entrenamiento = 100000
env_train = EntornoBancoBigData(X_train_scaled[:limite_entrenamiento], y_train[:limite_entrenamiento])

print(f"\n🧠 4. Entrenando al Supervisor Inteligente PPO (Esto tomará un par de minutos)...")
model_drl = PPO("MlpPolicy", env_train, verbose=0, learning_rate=0.0005)
model_drl.learn(total_timesteps=50000)
print("✅ ¡Entrenamiento del DRL completado!")

# ==============================================================================
# PASO 4: EVALUACIÓN MASIVA (TORNEO)
# ==============================================================================
print("\n🎯 5. Evaluando al Agente sobre el 20% de datos invisibles (Vectorizado)...")

# El DRL predice acciones (0, 1 o 2) para todas las transacciones de prueba de un golpe
acciones_drl, _ = model_drl.predict(X_test_scaled, deterministic=True)

# Convertimos las acciones del DRL a formato binario para comparar con XGBoost
# Si bloqueó (1) o pidió OTP (2), lo contamos como "Detección" (1). Si permitió (0), es (0).
y_pred_drl = (acciones_drl > 0).astype(int)

# Métricas
precision_drl, recall_curv_drl, _ = precision_recall_curve(y_test, y_pred_drl)
auprc_drl = auc(recall_curv_drl, precision_drl)
rec_drl = recall_score(y_test, y_pred_drl)
f1_drl = f1_score(y_test, y_pred_drl)
mcc_drl = matthews_corrcoef(y_test, y_pred_drl)

print("-" * 50)
print("🏆 RESULTADOS FINALES DE DRL (PISO 3):")
print("-" * 50)
print(f"🔸 AUPRC: {auprc_drl:.4f}")
print(f"🔸 Recall (Fraudes mitigados): {rec_drl:.2%}")
print(f"🔸 F1-Score: {f1_drl:.4f}")
print(f"🔸 MCC: {mcc_drl:.4f}")
print("-" * 50)

# Matriz de Confusión
cm_drl = confusion_matrix(y_test, y_pred_drl)
print("\n🚨 IMPACTO EN EL NEGOCIO (DRL):")
print(f"✔️ Clientes sanos permitidos sin fricción: {cm_drl[0][0]}")
print(f"⚠️ Falsos Positivos (Clientes sanos con OTP/Bloqueo): {cm_drl[0][1]}")
print(f"❌ Falsos Negativos (Fraudes que pasaron): {cm_drl[1][0]}")
print(f"🎯 FRAUDES DETECTADOS (Bloqueo o OTP): {cm_drl[1][1]}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 3.7 MB/s eta 0:00:00


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=

Mounted at /content/drive
🚀 1. Cargando matriz gigante para DRL...
⚙️ 2. Particionando y Escalando (Obligatorio para DRL)...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



🏦 3. Construyendo el Entorno de Simulacion Bancaria...

🧠 4. Entrenando al Supervisor Inteligente PPO (Esto tomará un par de minutos)...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✅ ¡Entrenamiento del DRL completado!

🎯 5. Evaluando al Agente sobre el 20% de datos invisibles (Vectorizado)...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


--------------------------------------------------
🏆 RESULTADOS FINALES DE DRL (PISO 3):
--------------------------------------------------
🔸 AUPRC: 0.5005
🔸 Recall (Fraudes mitigados): 0.00%
🔸 F1-Score: 0.0000
🔸 MCC: 0.0000
--------------------------------------------------

🚨 IMPACTO EN EL NEGOCIO (DRL):
✔️ Clientes sanos permitidos sin fricción: 294270
⚠️ Falsos Positivos (Clientes sanos con OTP/Bloqueo): 0
❌ Falsos Negativos (Fraudes que pasaron): 317
🎯 FRAUDES DETECTADOS (Bloqueo o OTP): 0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [2]:
# ==============================================================================
# PASO 3.1: EL ENTORNO BALANCEADO PARA DRL (Evitando el Reward Hacking)
# ==============================================================================
print("⚖️ 1. Creando un Simulador Balanceado (50% Fraude / 50% Normal)...")

# Separamos los fraudes y normales del set de entrenamiento
mask_fraude = (y_train == 1)
X_train_fraude = X_train_scaled[mask_fraude]
y_train_fraude = y_train[mask_fraude]

X_train_normal = X_train_scaled[~mask_fraude]
y_train_normal = y_train[~mask_fraude]

class EntornoBancoEquilibrado(gym.Env):
    def __init__(self, X_f, y_f, X_n, y_n):
        super(EntornoBancoEquilibrado, self).__init__()
        self.X_f = X_f
        self.y_f = y_f
        self.X_n = X_n
        self.y_n = y_n
        self.action_space = spaces.Discrete(3)
        self.observation_space = spaces.Box(low=0, high=1, shape=(X_f.shape[1],), dtype=np.float32)

    def _get_obs(self):
        # Magia: 50% de probabilidad de mostrar un fraude durante el entrenamiento
        if np.random.rand() > 0.5:
            idx = np.random.randint(len(self.X_f))
            self.current_y = self.y_f[idx]
            return self.X_f[idx]
        else:
            idx = np.random.randint(len(self.X_n))
            self.current_y = self.y_n[idx]
            return self.X_n[idx]

    def step(self, action):
        is_fraud = self.current_y

        # Castigos mucho más estrictos para obligarlo a pensar
        if is_fraud == 1:
            if action == 1: reward = 50     # ¡Excelente! Bloqueó el fraude
            elif action == 2: reward = 10   # Pasable: Pidió OTP
            else: reward = -100             # FATAL: Dejó pasar el fraude
        else:
            if action == 0: reward = 10     # ¡Excelente! Dejó pasar al cliente sano
            elif action == 2: reward = -5   # Leve: Molestó con OTP
            else: reward = -50              # GRAVE: Bloqueó a un cliente sano

        obs = self._get_obs()
        return obs, reward, False, False, {}

    def reset(self, seed=None):
        return self._get_obs(), {}

print("\n🧠 2. Re-entrenando al Agente PPO en el Simulador Balanceado...")
env_bal = EntornoBancoEquilibrado(X_train_fraude, y_train_fraude, X_train_normal, y_train_normal)

# Le damos 100,000 pasos de entrenamiento (Al ser aleatorio, es muy rápido)
model_drl_bal = PPO("MlpPolicy", env_bal, verbose=0, learning_rate=0.0003)
model_drl_bal.learn(total_timesteps=100000)
print("✅ ¡Entrenamiento completado!")

# ==============================================================================
# PASO 4: EVALUACIÓN EN EL MUNDO REAL (El 20% desbalanceado)
# ==============================================================================
print("\n🎯 3. Soltando al Agente entrenado en los datos de prueba reales...")

acciones_drl, _ = model_drl_bal.predict(X_test_scaled, deterministic=True)
y_pred_drl = (acciones_drl > 0).astype(int)

rec_drl = recall_score(y_test, y_pred_drl)
f1_drl = f1_score(y_test, y_pred_drl)
mcc_drl = matthews_corrcoef(y_test, y_pred_drl)

print("-" * 50)
print("🏆 RESULTADOS FINALES DE DRL (SIMULADOR BALANCEADO):")
print("-" * 50)
print(f"🔸 Recall (Fraudes mitigados): {rec_drl:.2%}")
print(f"🔸 F1-Score: {f1_drl:.4f}")
print(f"🔸 MCC: {mcc_drl:.4f}")
print("-" * 50)

cm_drl = confusion_matrix(y_test, y_pred_drl)
print("\n🚨 IMPACTO EN EL NEGOCIO (DRL):")
print(f"✔️ Clientes sanos permitidos sin fricción (Acción 0): {cm_drl[0][0]}")
print(f"⚠️ Falsos Positivos (Clientes sanos con OTP/Bloqueo): {cm_drl[0][1]}")
print(f"❌ Falsos Negativos (Fraudes que pasaron como si nada): {cm_drl[1][0]}")
print(f"🎯 FRAUDES DETECTADOS (Bloqueo o OTP): {cm_drl[1][1]}")

# Análisis de Option Learning
print("\n🔍 Desglose de Decisiones del Agente DRL:")
acciones_unicas, conteos = np.unique(acciones_drl, return_counts=True)
for accion, cantidad in zip(acciones_unicas, conteos):
    nombre = "Permitir (0)" if accion == 0 else "Bloquear (1)" if accion == 1 else "Pedir OTP (2)"
    print(f"   -> Decidió {nombre}: {cantidad} veces")

⚖️ 1. Creando un Simulador Balanceado (50% Fraude / 50% Normal)...

🧠 2. Re-entrenando al Agente PPO en el Simulador Balanceado...
✅ ¡Entrenamiento completado!

🎯 3. Soltando al Agente entrenado en los datos de prueba reales...
--------------------------------------------------
🏆 RESULTADOS FINALES DE DRL (SIMULADOR BALANCEADO):
--------------------------------------------------
🔸 Recall (Fraudes mitigados): 100.00%
🔸 F1-Score: 0.0022
🔸 MCC: 0.0019
--------------------------------------------------

🚨 IMPACTO EN EL NEGOCIO (DRL):
✔️ Clientes sanos permitidos sin fricción (Acción 0): 961
⚠️ Falsos Positivos (Clientes sanos con OTP/Bloqueo): 293309
❌ Falsos Negativos (Fraudes que pasaron como si nada): 0
🎯 FRAUDES DETECTADOS (Bloqueo o OTP): 317

🔍 Desglose de Decisiones del Agente DRL:
   -> Decidió Permitir (0): 961 veces
   -> Decidió Bloquear (1): 39757 veces
   -> Decidió Pedir OTP (2): 253869 veces


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# 1. IMPORTAR LIBRERÍAS
import pandas as pd
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from google.colab import drive

# 2. MONTAR DRIVE
drive.mount('/content/drive')

# 3. DEFINIR LA CLASE DEL ENTORNO (Si no la habías ejecutado)
class EntornoBanco(gym.Env):
    def __init__(self, df):
        super(EntornoBanco, self).__init__()
        self.df = df
        self.action_space = spaces.Discrete(3) # 0: Permitir, 1: Bloquear, 2: OTP
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(290,), dtype=np.float32)
        self.current_step = 0

    def step(self, action):
        row = self.df.iloc[self.current_step]
        is_fraud = row['FRAUDE_IA']
        # Lógica de recompensas
        if action == 1 and is_fraud == 1: reward = 10
        elif action == 1 and is_fraud == 0: reward = -5
        elif action == 0 and is_fraud == 1: reward = -20
        else: reward = 1
        self.current_step += 1
        done = self.current_step >= len(self.df) - 1
        # Usamos .values.astype(np.float32) para asegurar el formato correcto
        return self.df.iloc[self.current_step % len(self.df)].drop('FRAUDE_IA').values.astype(np.float32), reward, done, False, {}

    def reset(self, seed=None):
        self.current_step = 0
        return self.df.iloc[0].drop('FRAUDE_IA').values.astype(np.float32), {}

# 4. CARGAR DATOS Y ENTRENAR
print("🚀 Cargando y Entrenando...")
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/MAESTRIA IA/PROYECTO - DESARROLLO DE SOLUCIONES/dataset_limpio_IA.csv')
env = EntornoBanco(df.fillna(0))

model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=10000)
print("✅ ¡Entrenamiento completado!")

Mounted at /content/drive
🚀 Cargando y Entrenando...
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-----------------------------
| time/              |      |
|    fps             | 294  |
|    iterations      | 1    |
|    time_elapsed    | 6    |
|    total_timesteps | 2048 |
-----------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-----------------------------------------
| time/                   |             |
|    fps                  | 257         |
|    iterations           | 2           |
|    time_elapsed         | 15          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.013870352 |
|    clip_fraction        | 0.291       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.09       |
|    explained_variance   | 0.00122     |
|    learning_rate        | 0.0003      |
|    loss                 | 273         |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0347     |
|    value_loss           | 529         |
-----------------------------------------


...

-----------------------------------------
| time/                   |             |
|    fps                  | 273         |
|    iterations           | 4           |
|    time_elapsed         | 29          |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.011246957 |
|    clip_fraction        | 0.238       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1          |
|    explained_variance   | 0.00201     |
|    learning_rate        | 0.0003      |
|    loss                 | 203         |
|    n_updates            | 30          |
|    policy_gradient_loss | -0.0259     |
|    value_loss           | 313         |
-----------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-----------------------------------------
| time/                   |             |
|    fps                  | 284         |
|    iterations           | 5           |
|    time_elapsed         | 36          |
|    total_timesteps      | 10240       |
| train/                  |             |
|    approx_kl            | 0.014695152 |
|    clip_fraction        | 0.201       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.957      |
|    explained_variance   | -0.00622    |
|    learning_rate        | 0.0003      |
|    loss                 | 159         |
|    n_updates            | 40          |
|    policy_gradient_loss | -0.0257     |
|    value_loss           | 354         |
-----------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✅ ¡Entrenamiento completado!


In [ ]:
# 1. ENTORNO MEJORADO CON OPTION LEARNING (Jerárquico)
class EntornoBancoJerarquico(gym.Env):
    def __init__(self, df):
        super(EntornoBancoJerarquico, self).__init__()
        self.df = df
        self.action_space = spaces.Discrete(3) # 0: Permitir, 1: Bloquear, 2: OTP
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(290,), dtype=np.float32)
        self.current_step = 0

    def step(self, action):
        row = self.df.iloc[self.current_step]

        # --- OPTION LEARNING (Jerarquía por Canal) ---
        # El agente aprende que si el canal es 'Portal Virtual' (riesgoso),
        # la "opción" preferida es OTP, no bloqueo total.
        canal = row['Tipo_Transaccion_INGRESO PORTAL VIRTUAL'] if 'Tipo_Transaccion_INGRESO PORTAL VIRTUAL' in row else 0

        is_fraud = row['FRAUDE_IA']

        # Lógica Jerárquica: Si es canal crítico y fraude, la recompensa por OTP es mayor
        if is_fraud == 1:
            if canal == 1 and action == 2: reward = 15 # Acierto jerárquico: OTP en canal crítico
            elif action == 1: reward = 10              # Acierto: Bloqueo
            else: reward = -20
        else:
            reward = 1 if action == 0 else -2

        self.current_step += 1
        done = self.current_step >= len(self.df) - 1
        obs = self.df.iloc[self.current_step % len(self.df)].drop('FRAUDE_IA').values.astype(np.float32)
        return obs, reward, done, False, {}

    def reset(self, seed=None):
        self.current_step = 0
        return self.df.iloc[0].drop('FRAUDE_IA').values.astype(np.float32), {}

# 2. FUNCIÓN DE CONTINUAL LEARNING (Aprendizaje continuo)
def actualizar_modelo(modelo, nuevos_datos, env_actual):
    print("🔄 Ejecutando Continual Learning: Adaptando modelo a nuevos datos...")
    # Creamos un nuevo entorno temporal con los nuevos datos
    env_nuevos = EntornoBancoJerarquico(nuevos_datos.fillna(0))
    # 'Re-entrenamos' el modelo existente (sin borrar lo aprendido)
    modelo.set_env(env_nuevos)
    modelo.learn(total_timesteps=500) # Aprendizaje rápido para nuevos casos
    print("✅ Modelo actualizado con éxito.")
    return modelo

# --- ENTRENAMIENTO ---
env = EntornoBancoJerarquico(df_train_scaled)
model = PPO("MlpPolicy", env, verbose=0).learn(total_timesteps=10000)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Cargar datos
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/MAESTRIA IA/PROYECTO - DESARROLLO DE SOLUCIONES/dataset_limpio_IA.csv')
env = EntornoBanco(df.fillna(0))

# Crear el Agente (El "Cerebro")
model = PPO("MlpPolicy", env, verbose=1)

print("\n🚀 Entrenando al Supervisor Inteligente (DRL)...")
model.learn(total_timesteps=10000)
print("✅ ¡Entrenamiento completado!")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.

🚀 Entrenando al Supervisor Inteligente (DRL)...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

-----------------------------
| time/              |      |
|    fps             | 432  |
|    iterations      | 1    |
|    time_elapsed    | 4    |
|    total_timesteps | 2048 |
-----------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-----------------------------------------
| time/                   |             |
|    fps                  | 317         |
|    iterations           | 2           |
|    time_elapsed         | 12          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.013766093 |
|    clip_fraction        | 0.246       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.09       |
|    explained_variance   | -0.000777   |
|    learning_rate        | 0.0003      |
|    loss                 | 257         |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0293     |
|    value_loss           | 635         |
-----------------------------------------


...

------------------------------------------
| time/                   |              |
|    fps                  | 292          |
|    iterations           | 4            |
|    time_elapsed         | 27           |
|    total_timesteps      | 8192         |
| train/                  |              |
|    approx_kl            | 0.0153938895 |
|    clip_fraction        | 0.275        |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.998       |
|    explained_variance   | -0.0101      |
|    learning_rate        | 0.0003       |
|    loss                 | 149          |
|    n_updates            | 30           |
|    policy_gradient_loss | -0.0303      |
|    value_loss           | 245          |
------------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-----------------------------------------
| time/                   |             |
|    fps                  | 300         |
|    iterations           | 5           |
|    time_elapsed         | 34          |
|    total_timesteps      | 10240       |
| train/                  |             |
|    approx_kl            | 0.017222464 |
|    clip_fraction        | 0.145       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.946      |
|    explained_variance   | 0.00757     |
|    learning_rate        | 0.0003      |
|    loss                 | 207         |
|    n_updates            | 40          |
|    policy_gradient_loss | -0.0209     |
|    value_loss           | 323         |
-----------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✅ ¡Entrenamiento completado!


In [ ]:
# 1. PREPARACIÓN (Escalado de datos para que el DRL no explote)
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

# Separamos train y test
df_train, df_test = train_test_split(df.fillna(0), test_size=0.2, random_state=42)

# Escalamos todo (¡ESTO ES LO QUE ESTABA FALTANDO!)
X_train_scaled = scaler.fit_transform(df_train.drop(columns=['FRAUDE_IA']))
X_test_scaled = scaler.transform(df_test.drop(columns=['FRAUDE_IA']))

# Creamos DataFrames escalados
df_train_scaled = pd.DataFrame(X_train_scaled, columns=df_train.drop(columns=['FRAUDE_IA']).columns)
df_train_scaled['FRAUDE_IA'] = df_train['FRAUDE_IA'].values

df_test_scaled = pd.DataFrame(X_test_scaled, columns=df_test.drop(columns=['FRAUDE_IA']).columns)
df_test_scaled['FRAUDE_IA'] = df_test['FRAUDE_IA'].values

# 2. ENTRENAMIENTO
env_train = EntornoBanco(df_train_scaled)
model = PPO("MlpPolicy", env_train, verbose=0).learn(total_timesteps=10000)

# 3. VALIDACIÓN OPTIMIZADA (Vectorizada)
print("\n📊 Generando Matriz de Confusión para DRL (Versión Rápida)...")

# Convertimos todo el set de prueba a tensores de una vez
obs_test = df_test_scaled.drop(columns=['FRAUDE_IA']).values.astype(np.float32)

# El modelo PPO puede predecir todos los datos de un solo golpe
acciones, _ = model.predict(obs_test, deterministic=True)

# Generamos las predicciones
y_true_drl = df_test_scaled['FRAUDE_IA'].values
y_pred_drl = (acciones > 0).astype(int)

# 4. Imprimir Matriz
from sklearn.metrics import confusion_matrix
print("\nMatriz de Confusión Final:")
print(confusion_matrix(y_true_drl, y_pred_drl))


📊 Generando Matriz de Confusión para DRL (Versión Rápida)...

Matriz de Confusión Final:
[[24558  2749]
 [   63  8690]]


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from sklearn.metrics import precision_recall_curve, auc, f1_score, recall_score, matthews_corrcoef

# 1. Calculamos las métricas usando los vectores que ya tienes: y_true_drl y y_pred_drl
recall_drl = recall_score(y_true_drl, y_pred_drl)
f1_drl = f1_score(y_true_drl, y_pred_drl)
mcc_drl = matthews_corrcoef(y_true_drl, y_pred_drl)

# 2. AUPRC (Usando las probabilidades/acciones del DRL)
# En DRL, 'acciones' actúa como nuestro score de riesgo
precision, recall, _ = precision_recall_curve(y_true_drl, acciones)
auprc_drl = auc(recall, precision)

# 3. Imprimir el resumen idéntico al de XGBoost
print("-" * 50)
print("🏆 RESULTADOS FINALES DEL SUPERVISOR DRL (PISO 3):")
print("-" * 50)
print(f"🔸 AUPRC (Métrica Prioritaria): {auprc_drl:.4f}")
print(f"🔸 Recall (Fraudes detectados): {recall_drl:.2%}")
print(f"🔸 F1-Score: {f1_drl:.4f}")
print(f"🔸 Coeficiente de Matthews (MCC): {mcc_drl:.4f}")
print("-" * 50)

--------------------------------------------------
🏆 RESULTADOS FINALES DEL SUPERVISOR DRL (PISO 3):
--------------------------------------------------
🔸 AUPRC (Métrica Prioritaria): 0.7053
🔸 Recall (Fraudes detectados): 99.28%
🔸 F1-Score: 0.8607
🔸 Coeficiente de Matthews (MCC): 0.8219
--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
